# Exploring Cnes Data

## First look up at Data

In [3]:
from pyspark.sql import SparkSession
from pathlib import Path
import os
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="pyspark")


current_path = os.getcwd()
project_root = Path(current_path).parent
os.chdir(project_root)
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
spark = SparkSession.builder \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()
    
cnes_df = spark.read.csv("data/downloads/cnes_estabelecimentos.csv", header=True, inferSchema=True, sep=";")
cnes_df.show(5, truncate=False)

26/07/28 07:34:05 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-------------+-----+-------+-------------------+-----------------------------------------------+---------------------------------+-----------------------+-----------------------+---------+-------------------+-------------------+------------------------+------------------------+------------+----------+--------+-----------------------------+-----------+-------------------+------------+-------------+--------------+--------------------+----------------------------------------------------------------------------------+--------------+--------+---------------+-------------------+--------------------+------------------+-------------------+----------------+---------------------+---------------+-------------------+
|CO_CNES|CO_UNIDADE   |CO_UF|CO_IBGE|NU_CNPJ_MANTENEDORA|NO_RAZAO_SOCIAL                                |NO_FANTASIA                      |CO_NATUREZA_ORGANIZACAO|DS_NATUREZA_ORGANIZACAO|TP_GESTAO|CO_NIVEL_HIERARQUIA|DS_NIVEL_HIERARQUIA|CO_ESFERA_ADMINISTRATIVA|DS_ESFERA_ADMI

In [5]:
cnes_df.printSchema()

root
 |-- CO_CNES: integer (nullable = true)
 |-- CO_UNIDADE: string (nullable = true)
 |-- CO_UF: integer (nullable = true)
 |-- CO_IBGE: integer (nullable = true)
 |-- NU_CNPJ_MANTENEDORA: long (nullable = true)
 |-- NO_RAZAO_SOCIAL: string (nullable = true)
 |-- NO_FANTASIA: string (nullable = true)
 |-- CO_NATUREZA_ORGANIZACAO: integer (nullable = true)
 |-- DS_NATUREZA_ORGANIZACAO: string (nullable = true)
 |-- TP_GESTAO: string (nullable = true)
 |-- CO_NIVEL_HIERARQUIA: integer (nullable = true)
 |-- DS_NIVEL_HIERARQUIA: string (nullable = true)
 |-- CO_ESFERA_ADMINISTRATIVA: string (nullable = true)
 |-- DS_ESFERA_ADMINISTRATIVA: string (nullable = true)
 |-- CO_ATIVIDADE: integer (nullable = true)
 |-- TP_UNIDADE: integer (nullable = true)
 |-- CO_CEP: integer (nullable = true)
 |-- NO_LOGRADOURO: string (nullable = true)
 |-- NU_ENDERECO: string (nullable = true)
 |-- NO_BAIRRO: string (nullable = true)
 |-- NU_TELEFONE: string (nullable = true)
 |-- NU_LATITUDE: double (null

In [6]:
cnes_df.count()

628398

## Preenchimento campos

In [4]:
from pyspark.sql import functions as F

total_rows = cnes_df.count()

fulfillment_exprs = [
    F.struct(
        F.lit(c).alias("column"),
        (F.count(F.col(c)) / F.lit(total_rows) * 100).alias("fulfillment_pct")
    )
    for c in cnes_df.columns
]

result = cnes_df.select(F.array(*fulfillment_exprs).alias("stats")) \
    .selectExpr("explode(stats) as stats") \
    .select("stats.column", "stats.fulfillment_pct")

result.orderBy(F.col("fulfillment_pct").desc()).show(cnes_df.columns.__len__(), truncate=False)

+------------------------+--------------------+
|column                  |fulfillment_pct     |
+------------------------+--------------------+
|CO_CNES                 |100.0               |
|CO_UNIDADE              |100.0               |
|CO_UF                   |100.0               |
|CO_IBGE                 |100.0               |
|TP_GESTAO               |100.0               |
|CO_ESFERA_ADMINISTRATIVA|100.0               |
|DS_ESFERA_ADMINISTRATIVA|100.0               |
|CO_ATIVIDADE            |100.0               |
|TP_UNIDADE              |100.0               |
|CO_CEP                  |100.0               |
|NO_LOGRADOURO           |100.0               |
|ST_SERVICO_APOIO        |100.0               |
|ST_ATEND_AMBULATORIAL   |100.0               |
|CO_AMBULATORIAL_SUS     |100.0               |
|NO_RAZAO_SOCIAL         |99.99984086518417   |
|NO_BAIRRO               |99.99984086518417   |
|CO_NATUREZA_JUR         |99.99936346073667   |
|NO_FANTASIA             |99.99283893328